In [41]:
import json
import os
from pathlib import Path

import ipyfilechooser as FC
import ipywidgets as W
from IPython.display import clear_output, display

In [47]:
# Header
header = W.HTML("""
        <div style='background-color: #DFF2FD; color: #006699; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 20px;'>
            <h1>SysSimX: System Simulation Framework Interface</h1>
            <h2>Demo User Interface</h2>
        </div>
        """)

# Mode Selection
mode_toggle = W.ToggleButtons(
    options=[("Standard", "standard"), ("Expert", "expert")],
    value="standard",
    description="User Mode:",
    style={"description_width": "initial", "button_width": "150px"},
    layout=W.Layout(width="400px"),
)

mode_section = W.VBox(
    [W.HTML("<h3>User Experience Level</h3>"), mode_toggle],
    layout=W.Layout(margin="10px", padding="10px", border="1px solid #ddd"),
)

# File chooser
file_chooser = FC.FileChooser(
    os.getcwd(),
    title="Select Model File",
    description="Choose a model file to load:",
    show_hidden=False,
    use_dir_icons=True,
    layout=W.Layout(width="600px", height="250px"),
)

# Save Button
save_button = W.Button(
    description="Add Selected File",
    button_style="success",
    tooltip="Add the selected file to the list",
    icon="check",
)

# Clear Button
clear_button = W.Button(
    description="Clear File List",
    button_style="warning",
    tooltip="Clear the list of selected files",
    icon="trash",
)

# Text Area to display selected files
file_list_area = W.Textarea(
    value="",
    placeholder="Selected files will appear here...",
    description="Selected Files:",
    layout=W.Layout(width="600px", height="150px"),
    disabled=True,
)

# Container for file selection section
file_selection_section = W.VBox(
    [
        W.HTML("<h3>Model File Selection</h3>"),
        file_chooser,
        W.HBox([save_button, clear_button]),
        file_list_area,
    ],
    layout=W.Layout(margin="10px", padding="10px", border="1px solid #ddd"),
)

# Complete UI
ui = W.VBox([header, mode_section, file_selection_section])

In [ ]:
# Function which adds a selected file to a list (for demonstration)
FILE_LIST = set()


def on_file_selected(chooser: FC.FileChooser):
    selected_file = chooser.selected
    if selected_file:
        FILE_LIST.add(selected_file)


# Add a save button to confirm file selection and append to FILE_LIST


def on_save_button_clicked(b):
    on_file_selected(file_chooser)


save_button.on_click(on_save_button_clicked)
# display(save_button)

# Add a clear button to reset the FILE_LIST


def on_clear_button_clicked(b):
    FILE_LIST.clear()
    print("File list cleared.")


clear_button.on_click(on_clear_button_clicked)


# Update display of selected files
def update_file_display():
    clear_output(wait=True)
    display(header)
    # display(mode_section)
    display(file_chooser)
    display(save_button)
    display(clear_button)
    if FILE_LIST:
        print("Selected Files:")
        for f in FILE_LIST:
            print(f" - {f}")


update_file_display()
save_button.on_click(lambda b: update_file_display())
clear_button.on_click(lambda b: update_file_display())

HTML(value="\n        <div style='background-color: #DFF2FD; color: #006699; padding: 20px; border-radius: 10p…

FileChooser(path='/home/flo/repos/SystemSimulation/examples/UI', filename='', title='Select Model File', show_…

Button(button_style='success', description='Add Selected File', icon='check', style=ButtonStyle(), tooltip='Ad…

Button(button_style='warning', description='Clear File List', icon='trash', style=ButtonStyle(), tooltip='Clea…

In [40]:
FILE_LIST

{'/home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/AngleEncoder.mo'}

In [21]:
on_file_selected(file_chooser)

Selected file: /home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/AngleEncoder.mo


---------------------

In [2]:
CONFIG_PATH = Path("sim_config_step1.json")

In [3]:
mode_toggle = W.ToggleButtons(
    options=[("Standard user", "standard"), ("Expert user", "expert")],
    value="standard",
    description="Mode:",
)

In [4]:
pkg_location = FC.FileChooser(
    title="Select package location",
    description="Package location:",
    show_hidden=False,
    select_default=True,
    use_dir_icons=True,
    width="100%",
)
pkg_location.default_filename = "my_simulation_package"
pkg_location.register_callback(
    lambda chooser: print(f"Selected package location: {chooser.selected_path}")
)

display(mode_toggle, pkg_location)

ToggleButtons(description='Mode:', options=(('Standard user', 'standard'), ('Expert user', 'expert')), value='…

FileChooser(path='/home/flo/repos/SystemSimulation/examples/UI', filename='my_simulation_package', title='Sele…

In [10]:
# Load files from directory and save paths in lists
from pathlib import Path


def load_files_from_directory(directory_path, file_extensions=None, recursive=False):
    """
    Load files from directory and return their paths as a list.

    Args:
        directory_path: Path to directory to scan
        file_extensions: List of extensions to filter (e.g., ['.mo', '.py'])
        recursive: Whether to search subdirectories recursively

    Returns:
        List of Path objects for found files
    """
    directory = Path(directory_path)

    if not directory.exists() or not directory.is_dir():
        print(f"Directory {directory} does not exist or is not a directory")
        return []

    file_paths = []

    if recursive:
        # Search recursively using **/* pattern
        if file_extensions:
            for ext in file_extensions:
                file_paths.extend(directory.rglob(f"*{ext}"))
        else:
            # Get all files recursively
            file_paths.extend([f for f in directory.rglob("*") if f.is_file()])
    else:
        # Search only in current directory
        if file_extensions:
            for ext in file_extensions:
                file_paths.extend(directory.glob(f"*{ext}"))
        else:
            # Get all files in current directory
            file_paths.extend([f for f in directory.iterdir() if f.is_file()])

    return sorted(file_paths)


# Get the selected package directory
if pkg_location.selected_path:
    pkg_dir = Path(pkg_location.selected_path)
else:
    # Default to examples/Modelica if no selection
    pkg_dir = Path.cwd().parent.parent / "examples" / "Modelica"

print(f"Loading files from: {pkg_dir}")

# Load Modelica files (.mo) from directory
modelica_files = load_files_from_directory(pkg_dir, file_extensions=[".mo"], recursive=True)
print(f"Found {len(modelica_files)} Modelica files:")

# Create lists to store file paths
modelica_file_paths = []  # List of Path objects
modelica_file_names = []  # List of file names only
modelica_relative_paths = []  # List of paths relative to package directory

for file_path in modelica_files:
    modelica_file_paths.append(file_path)
    modelica_file_names.append(file_path.name)

    # Get relative path from package directory
    try:
        rel_path = file_path.relative_to(pkg_dir)
        modelica_relative_paths.append(str(rel_path))
    except ValueError:
        modelica_relative_paths.append(str(file_path))

# Display found files
for i, (path, rel_path) in enumerate(zip(modelica_file_paths, modelica_relative_paths)):
    print(f"  {i + 1:2d}. {rel_path}")

# Create dropdown with all found files
if modelica_files:
    mo_file_dropdown = W.Dropdown(
        options=[
            (rel_path, full_path)
            for rel_path, full_path in zip(modelica_relative_paths, modelica_file_paths)
        ],
        description="Model file:",
        style={"description_width": "initial"},
        layout=W.Layout(width="70%"),
    )

    display(mo_file_dropdown)

    # Show the content of the selected .mo file
    def show_mo_file_content(change):
        if change["type"] == "change" and change["name"] == "value":
            selected_file = change["new"]
            if selected_file and selected_file.exists():
                try:
                    with open(selected_file, encoding="utf-8") as f:
                        content = f.read()
                    mo_file_content.value = content
                except Exception as e:
                    mo_file_content.value = f"Error reading file: {e}"
            else:
                mo_file_content.value = "File does not exist."

    mo_file_dropdown.observe(show_mo_file_content)
    mo_file_content = W.Textarea(
        value="Select a file to view its content...",
        description="File content:",
        layout=W.Layout(width="100%", height="400px"),
        disabled=True,
    )
    display(mo_file_content)

    # Trigger initial display
    if mo_file_dropdown.options:
        show_mo_file_content({"type": "change", "name": "value", "new": mo_file_dropdown.value})
else:
    print("No Modelica files found in the selected directory.")

# Print the lists for debugging/verification
print("\n📋 Summary:")
print(f"  • Total files found: {len(modelica_file_paths)}")
print(
    f"  • File paths list: modelica_file_paths (contains {len(modelica_file_paths)} Path objects)"
)
print(f"  • File names list: modelica_file_names (contains {len(modelica_file_names)} strings)")
print(
    f"  • Relative paths list: modelica_relative_paths (contains {len(modelica_relative_paths)} strings)"
)

Loading files from: /home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum
Found 14 Modelica files:
   1. AngleEncoder.mo
   2. Demo_Driven.mo
   3. Demo_DrivenWithWall.mo
   4. Demo_DrivenWithWallDiscrete.mo
   5. Demo_UndrivenWithWall.mo
   6. Drive.mo
   7. ImpactWall.mo
   8. PID_Continuous.mo
   9. PID_Sampled.mo
  10. Pendulum.mo
  11. PendulumWithWallContinuous.mo
  12. PendulumWithWallDiscrete.mo
  13. Reference.mo
  14. package.mo


Dropdown(description='Model file:', layout=Layout(width='70%'), options=(('AngleEncoder.mo', PosixPath('/home/…

Textarea(value='Select a file to view its content...', description='File content:', disabled=True, layout=Layo…


📋 Summary:
  • Total files found: 14
  • File paths list: modelica_file_paths (contains 14 Path objects)
  • File names list: modelica_file_names (contains 14 strings)
  • Relative paths list: modelica_relative_paths (contains 14 strings)


In [11]:
# Example: Working with the loaded file paths
print("🔍 Examples of using the file path lists:")
print()

# 1. Access files by index
if modelica_file_paths:
    print("1️⃣ Access files by index:")
    for i in range(min(3, len(modelica_file_paths))):  # Show first 3 files
        file_path = modelica_file_paths[i]
        print(f"   File {i}: {file_path.name}")
        print(f"   Full path: {file_path}")
        print(f"   Size: {file_path.stat().st_size} bytes")
        print()

# 2. Filter files by pattern
print("2️⃣ Filter files by pattern:")
pendulum_files = [path for path in modelica_file_paths if "pendulum" in path.name.lower()]
filter_files = [path for path in modelica_file_paths if "filter" in path.name.lower()]

print(f"   Pendulum files: {len(pendulum_files)}")
for file in pendulum_files[:2]:  # Show first 2
    print(f"     • {file.name}")

print(f"   Filter files: {len(filter_files)}")
for file in filter_files[:2]:  # Show first 2
    print(f"     • {file.name}")

# 3. Convert paths to strings
print("\n3️⃣ Convert paths to strings:")
string_paths = [str(path) for path in modelica_file_paths[:3]]
for i, str_path in enumerate(string_paths):
    print(f"   String {i}: {str_path}")

# 4. Group files by directory
print("\n4️⃣ Group files by directory:")
from collections import defaultdict

files_by_dir = defaultdict(list)

for file_path in modelica_file_paths:
    parent_dir = file_path.parent.name
    files_by_dir[parent_dir].append(file_path.name)

for dir_name, files in list(files_by_dir.items())[:3]:  # Show first 3 directories
    print(f"   Directory '{dir_name}': {len(files)} files")
    for file in files[:2]:  # Show first 2 files per directory
        print(f"     • {file}")

# 5. Save paths to different formats
print("\n5️⃣ Save file paths in different formats:")

# As simple list of strings
paths_as_strings = [str(path) for path in modelica_file_paths]
print(f"   Paths as strings: {len(paths_as_strings)} items")

# As dictionary with metadata
files_metadata = []
for path in modelica_file_paths[:3]:  # Show first 3
    metadata = {
        "name": path.name,
        "full_path": str(path),
        "relative_path": str(path.relative_to(pkg_dir))
        if path.is_relative_to(pkg_dir)
        else str(path),
        "size_bytes": path.stat().st_size,
        "parent_dir": path.parent.name,
    }
    files_metadata.append(metadata)

print(f"   Files with metadata: {len(files_metadata)} items")
for i, meta in enumerate(files_metadata):
    print(f"     File {i}: {meta['name']} ({meta['size_bytes']} bytes)")

print("\n✅ File paths successfully loaded into lists!")
print("📊 Use 'modelica_file_paths' for Path objects")
print("📊 Use 'modelica_file_names' for file names only")
print("📊 Use 'modelica_relative_paths' for relative paths")

🔍 Examples of using the file path lists:

1️⃣ Access files by index:
   File 0: AngleEncoder.mo
   Full path: /home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/AngleEncoder.mo
   Size: 1096 bytes

   File 1: Demo_Driven.mo
   Full path: /home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/Demo_Driven.mo
   Size: 1253 bytes

   File 2: Demo_DrivenWithWall.mo
   Full path: /home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/Demo_DrivenWithWall.mo
   Size: 1848 bytes

2️⃣ Filter files by pattern:
   Pendulum files: 3
     • Pendulum.mo
     • PendulumWithWallContinuous.mo
   Filter files: 0

3️⃣ Convert paths to strings:
   String 0: /home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/AngleEncoder.mo
   String 1: /home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/Demo_Driven.mo
   String 2: /home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendul

In [12]:
# Save file paths to different formats
import csv

print("💾 Saving file paths to different formats:")

if modelica_file_paths:
    # 1. Save as JSON
    json_data = {
        "scan_info": {
            "directory": str(pkg_dir),
            "total_files": len(modelica_file_paths),
            "scan_date": str(Path(__file__).stat().st_mtime)
            if "__file__" in globals()
            else "notebook",
        },
        "files": [],
    }

    for path in modelica_file_paths:
        file_info = {
            "name": path.name,
            "full_path": str(path),
            "relative_path": str(path.relative_to(pkg_dir))
            if path.is_relative_to(pkg_dir)
            else str(path),
            "size_bytes": path.stat().st_size,
            "directory": path.parent.name,
        }
        json_data["files"].append(file_info)

    # Save JSON file
    json_file = Path("modelica_files.json")
    with open(json_file, "w") as f:
        json.dump(json_data, f, indent=2)
    print(f"✅ Saved JSON: {json_file} ({len(json_data['files'])} files)")

    # 2. Save as CSV
    csv_file = Path("modelica_files.csv")
    with open(csv_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Name", "Full Path", "Relative Path", "Size (bytes)", "Directory"])

        for path in modelica_file_paths:
            relative = str(path.relative_to(pkg_dir)) if path.is_relative_to(pkg_dir) else str(path)
            writer.writerow([path.name, str(path), relative, path.stat().st_size, path.parent.name])

    print(f"✅ Saved CSV: {csv_file} ({len(modelica_file_paths)} files)")

    # 3. Save as simple text list
    txt_file = Path("modelica_files.txt")
    with open(txt_file, "w") as f:
        f.write(f"Modelica Files in {pkg_dir}\n")
        f.write("=" * 50 + "\n\n")
        for i, path in enumerate(modelica_file_paths, 1):
            f.write(f"{i:3d}. {path.name}\n")
            f.write(f"     Path: {path}\n")
            f.write(f"     Size: {path.stat().st_size} bytes\n\n")

    print(f"✅ Saved TXT: {txt_file} ({len(modelica_file_paths)} files)")

    # 4. Create Python list variables for direct use
    print("\n🐍 Python variables created:")
    print(f"   • modelica_file_paths: List[Path] with {len(modelica_file_paths)} items")
    print(f"   • modelica_file_names: List[str] with {len(modelica_file_names)} items")
    print(f"   • modelica_relative_paths: List[str] with {len(modelica_relative_paths)} items")

    # Example usage
    print("\n🔧 Example usage:")
    print("   # Get first file path")
    print("   first_file = modelica_file_paths[0]")
    if modelica_file_paths:
        print(f"   # Result: {modelica_file_paths[0]}")

    print("\n   # Get all file names")
    print("   all_names = modelica_file_names")
    print(f"   # Result: {modelica_file_names[:3]}{'...' if len(modelica_file_names) > 3 else ''}")

    print("\n   # Filter by extension or pattern")
    print("   mo_files = [p for p in modelica_file_paths if p.suffix == '.mo']")
    mo_only = [p for p in modelica_file_paths if p.suffix == ".mo"]
    print(f"   # Result: {len(mo_only)} .mo files")

else:
    print("❌ No files found to save")

print("\n✅ File path loading and saving complete!")

💾 Saving file paths to different formats:
✅ Saved JSON: modelica_files.json (14 files)
✅ Saved CSV: modelica_files.csv (14 files)
✅ Saved TXT: modelica_files.txt (14 files)

🐍 Python variables created:
   • modelica_file_paths: List[Path] with 14 items
   • modelica_file_names: List[str] with 14 items
   • modelica_relative_paths: List[str] with 14 items

🔧 Example usage:
   # Get first file path
   first_file = modelica_file_paths[0]
   # Result: /home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/AngleEncoder.mo

   # Get all file names
   all_names = modelica_file_names
   # Result: ['AngleEncoder.mo', 'Demo_Driven.mo', 'Demo_DrivenWithWall.mo']...

   # Filter by extension or pattern
   mo_files = [p for p in modelica_file_paths if p.suffix == '.mo']
   # Result: 14 .mo files

✅ File path loading and saving complete!


In [9]:
# Widget to display selected models
relevant_models = W.SelectMultiple(
    options=[f.name for f in mo_files],
    description="Relevant models:",
    style={"description_width": "initial"},
    layout=W.Layout(width="50%", height="150px"),
)

selected_models = W.Select(
    options=[],
    description="Selected models:",
    style={"description_width": "initial"},
    layout=W.Layout(width="50%", height="150px"),
)


def update_selected_models(change):
    selected_models.options = list(change["new"])


relevant_models.observe(update_selected_models, names="value")

display(relevant_models, selected_models)

SelectMultiple(description='Relevant models:', layout=Layout(height='150px', width='50%'), options=(), style=D…

Select(description='Selected models:', layout=Layout(height='150px', width='50%'), options=(), style=Descripti…

In [7]:
pkg_location = W.Text(
    value="",
    description="Package Path:",
    placeholder="Optional: path to Modelica package directory",
    layout=W.Layout(width="60%"),
)
model_file = W.Text(
    value="",
    description="Model File:",
    placeholder="Path to .mo file (e.g., /path/to/MySystem.mo)",
    layout=W.Layout(width="60%"),
)
model_class = W.Text(
    value="",
    description="Model Class:",
    placeholder="e.g., MyLib.Subsystems.Pendulum",
    layout=W.Layout(width="60%"),
)

use_mode = W.Dropdown(
    options=[("Run as Modelica (direct)", "modelica"), ("Export as FMU", "fmu")],
    value="modelica",
    description="Use Model:",
)

fmi_version = W.Dropdown(
    options=[("FMI 3.0", "3.0"), ("FMI 2.0", "2.0")],
    value="3.0",
    description="FMI Version:",
)
fmu_type = W.Dropdown(
    options=[("Model Exchange (ME)", "ME"), ("Co-Simulation (CS)", "CS")],
    value="ME",
    description="FMU Type:",
)
binary_format = W.Dropdown(
    options=[("Native", "native"), ("x86_64", "x86_64"), ("arm64", "arm64")],
    value="native",
    description="Binary:",
)
compiler_flags = W.Textarea(
    value="",
    placeholder="Extra compiler flags (optional)",
    description="Compiler Flags:",
    layout=W.Layout(width="60%", height="60px"),
)

expert_box = W.VBox(
    [
        W.HTML("<b>Advanced FMU Options</b>"),
        W.HBox([fmi_version, fmu_type, binary_format]),
        compiler_flags,
    ]
)

status = W.HTML("<i>Fill in the fields to continue…</i>", layout=W.Layout(margin="8px 0"))
problems = W.HTML("", layout=W.Layout(margin="0 0 8px 0"))


def validate():
    msgs = []
    if not model_file.value.strip():
        msgs.append("• Missing <b>Model File</b> (path to .mo).")
    if not model_class.value.strip():
        msgs.append("• Missing <b>Model Class</b> (e.g., MyLib.Subsystems.Pendulum).")
    if msgs:
        problems.value = "<div style='color:#b91c1c'>" + "<br>".join(msgs) + "</div>"
        status.value = "<span style='color:#b45309'>Please complete the required fields.</span>"
        return False
    else:
        problems.value = ""
        status.value = "<span style='color:#065f46'>Looks good. You can save this step.</span>"
        return True


for w in [
    pkg_location,
    model_file,
    model_class,
    use_mode,
    fmi_version,
    fmu_type,
    binary_format,
    compiler_flags,
]:
    w.observe(lambda change: validate(), names="value")


def on_mode_change(change):
    expert_box.layout.display = "none" if mode_toggle.value == "standard" else "block"
    validate()


mode_toggle.observe(on_mode_change, names="value")
on_mode_change(None)

save_btn = W.Button(description="Save Step 1", icon="save", button_style="success")
save_out = W.Output()


def on_save_clicked(b):
    save_out.clear_output()
    if not validate():
        with save_out:
            display(
                HTML("<div style='color:#b91c1c'>Cannot save: please fix the issues above.</div>")
            )
        return
    cfg = {
        "user_mode": mode_toggle.value,
        "model": {
            "package_path": pkg_location.value.strip() or None,
            "model_file": model_file.value.strip(),
            "model_class": model_class.value.strip(),
            "use_mode": use_mode.value,
        },
        "fmu_options": None,
    }
    if use_mode.value == "fmu":
        cfg["fmu_options"] = {
            "fmi_version": fmi_version.value,
            "fmu_type": fmu_type.value,
            "binary": binary_format.value,
            "compiler_flags": compiler_flags.value.strip() or None,
        }
    CONFIG_PATH.write_text(json.dumps(cfg, indent=2))
    with save_out:
        display(
            HTML(
                f"<div style='color:#065f46'>Saved Step 1 configuration.</div>"
                f"<div>Saved: <code>{CONFIG_PATH.resolve()}</code></div>"
            )
        )


save_btn.on_click(on_save_clicked)

panel = W.VBox(
    [
        W.HTML("<h3>Step 1 — Choose Modelica Model & Usage</h3>"),
        mode_toggle,
        W.HTML("<hr>"),
        W.HTML("<b>Modelica Source</b>"),
        pkg_location,
        model_file,
        model_class,
        W.HTML("<b>Usage</b>"),
        use_mode,
        expert_box,
        W.HTML("<hr>"),
        status,
        problems,
        W.HBox([save_btn]),
        save_out,
    ]
)

display(panel)

## 2) Next steps (suggested)

- **Step 2 — Solver & Simulation Settings**: ask for global time settings, tolerance, stop time; in Expert mode expose solver choices, step-size policy, event handling.
- **Step 3 — Build/Export**: if FMU selected, run OM → FMU export and show logs. If Modelica-only, compile and prepare simulation.
- **Step 4 — Run**: execute simulation with a progress log; offer live plots for selected signals.
- **Step 5 — Results**: store a run folder with metadata (config JSON/YAML, logs), plots, and CSV/HDF5 outputs.